# 02e — High-resolution retrain (yolov8s @ 960)

Targets the two measured weaknesses of `baseline_yolov8s` (30 ep @ 640, test mAP50 0.2472 @ conf=0.25):

- **Small objects** — recall@0.5 was 0.405 small vs 0.736 large, and 24,104 of 30,832 test boxes are small. Training and inferring at 960 instead of 640 gives every box ~2.25x the pixels, which is the single biggest lever available without new data.
- **The class tail** — 46 of 102 evaluable classes at AP=0. `copy_paste` augmentation pastes object instances across images, which disproportionately helps rare classes and small objects.

Budget guardrails, since this must fit one Colab T4 session:

- `time=5.5` hard-caps training at 5.5 wall-clock hours no matter what `epochs` says; Ultralytics stops on schedule and keeps `best.pt`.
- Warm-started from `baseline_yolov8s/weights/best.pt` (already knows the 148 classes) instead of COCO weights, so the run spends its limited hours adapting to 960px rather than relearning the taxonomy.
- Runs write directly to `RUNS_DIR` on Drive, so a dead session loses nothing; re-running the train cell auto-resumes from `last.pt`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 7.2 MB/s eta 0:00:00


In [2]:
import sys
from pathlib import Path

SCRIPTS_DIR = Path("/content/drive/Shareddrives/Computer Vision Final Project/Modeling/YOLO")
sys.path.insert(0, str(SCRIPTS_DIR))

from yolo_common_colab import *

# Extracts the dataset tar to local disk on first call (~5-25 min depending on Drive throughput).
data_yaml_path, data_yaml, unified_classes, train_image_paths, val_image_paths, test_image_paths = load_dataset()
print(f"{len(train_image_paths)} train / {len(val_image_paths)} val / {len(test_image_paths)} test")


Extracting /content/drive/Shareddrives/Computer Vision Final Project/Data/tar_datasets/fv40_merged.tar (3017 MB) -> /content/dataset...
34929 entries in the tar
  2000/34929  (28 files/s, ~19.4 min left)
  4000/34929  (55 files/s, ~9.3 min left)
  6000/34929  (82 files/s, ~5.8 min left)
  8000/34929  (108 files/s, ~4.2 min left)
  10000/34929  (134 files/s, ~3.1 min left)
  12000/34929  (160 files/s, ~2.4 min left)
  14000/34929  (186 files/s, ~1.9 min left)
  16000/34929  (212 files/s, ~1.5 min left)
  18000/34929  (220 files/s, ~1.3 min left)
  20000/34929  (244 files/s, ~1.0 min left)
  22000/34929  (268 files/s, ~0.8 min left)
  24000/34929  (291 files/s, ~0.6 min left)
  26000/34929  (315 files/s, ~0.5 min left)
  28000/34929  (339 files/s, ~0.3 min left)
  30000/34929  (362 files/s, ~0.2 min left)
  32000/34929  (385 files/s, ~0.1 min left)
  34000/34929  (409 files/s, ~0.0 min left)
  34929/34929  (419 files/s, ~0.0 min left)
Extraction complete in 1.4 min.
Extracting /content/d

In [3]:
from ultralytics import YOLO

RUN_NAME = "highres_yolov8s_960"
run_dir = RUNS_DIR / RUN_NAME
last_ckpt = run_dir / "weights" / "last.pt"
baseline_ckpt = RUNS_DIR / "baseline_yolov8s" / "weights" / "best.pt"

# If a previous session died mid-run, pick up where it left off; otherwise warm-start
# from the 640px baseline so the limited T4 hours go to adapting, not relearning.
resuming = last_ckpt.exists()
model = YOLO(str(last_ckpt if resuming else baseline_ckpt))
print(f"{'Resuming' if resuming else 'Warm-starting'} from {last_ckpt if resuming else baseline_ckpt}")

results = model.train(
    data=str(data_yaml_path),
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    resume=resuming,

    epochs=60,          # upper bound; the time cap below is what actually ends the run
    time=5.5,           # hard wall-clock cap in hours -- leaves ~2h of the session for eval
    patience=15,

    imgsz=960,
    batch=-1,           # auto-fit the T4's 15GB (expect ~8-12 at 960 for v8s)

    copy_paste=0.3,     # instance paste-in: the tail-class / small-object lever
    mixup=0.1,
    close_mosaic=10,    # last 10 epochs on clean images, as in the baseline recipe

    seed=42,
    plots=True,
)


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Warm-starting from /content/drive/Shareddrives/Computer Vision Final Project/Data/yolo_runs_colab_tar/baseline_yolov8s/weights/best.pt
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data_local.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None,

## Test-set evaluation

Two numbers on purpose:

1. **`conf=0.25` protocol** — directly comparable to the 0.2472 / 0.1461 the baseline reported in `02d` and `MODEL_SELECTION.md`. This is the swap-in decision number.
2. **`conf=0.001` protocol** — standard COCO-style scoring (the baseline's 640px checkpoint was never re-scored this way on test; its val-split number was 0.366). Report this as the honest headline metric for both models.


In [4]:
best = YOLO(str(run_dir / "weights" / "best.pt"))

print("=== conf=0.25 (baseline-comparable protocol) ===")
m = best.val(data=str(data_yaml_path), split="test", imgsz=960, conf=0.25, iou=0.45)
print(f"mAP@50: {m.box.map50:.4f}   mAP@50:95: {m.box.map:.4f}   P: {m.box.mp:.4f}   R: {m.box.mr:.4f}")

print("\n=== conf=0.001 (standard COCO-style protocol) ===")
m2 = best.val(data=str(data_yaml_path), split="test", imgsz=960, conf=0.001, iou=0.45)
print(f"mAP@50: {m2.box.map50:.4f}   mAP@50:95: {m2.box.map:.4f}")

# Decision rule: swap into the app if the conf=0.25 numbers beat 0.2472 / 0.1461.


=== conf=0.25 (baseline-comparable protocol) ===
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 11,182,860 parameters, 0 gradients, 28.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1462.8±784.3 MB/s, size: 69.9 KB)
val: Scanning /content/dataset/food_ingredients/test/labels... 1902 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1902/1902 1.5Kit/s 1.3s
val: New cache created: /content/dataset/food_ingredients/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 7.3it/s 16.4s
                   all       1902      30832      0.397      0.285      0.251       0.15
                almond          2         56      0.844      0.679      0.638      0.437
                 apple        308       2165      0.509      0.642      0.545      0.454
               apricot          1          4          0          0     

## Export for the app

Only run after the eval above beats the baseline. The app loads `best.pt` via Ultralytics directly, so copying that file down is sufficient; the ONNX export is for parity with the baseline's deployment artifact. Note `imgsz=960` — the app's detector must infer at the resolution the model was trained at.


In [5]:
export_path = best.export(format="onnx", imgsz=960, dynamic=True)
print(f"Exported: {export_path}")
print(f"Weights to download for the app: {run_dir / 'weights' / 'best.pt'}")


Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/drive/Shareddrives/Computer Vision Final Project/Data/yolo_runs_colab_tar/highres_yolov8s_960/weights/best.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) (1, 152, 18900) (21.6 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 332ms
Prepared 4 packages in 1.46s
Installed 4 packages in 252ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 2.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with